# 03. Модуль 3: VAE на мел-спектрограммах

Speech Commands: MSE и L1, β = 1 и β = 4, условный VAE. VAE на окнах русской речи (Dusha) даёт признаки для модуля 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!test -d /content/call-audio-analysis || git clone -q https://github.com/tanyalebedeva/call-audio-analysis.git /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
REPORTS = '/content/drive/MyDrive/coursework/reports'
!mkdir -p $DATA $RUNS $REPORTS
!nvidia-smi -L

## Обучение

In [ ]:
!python -m callaudio.train --config configs/vae/vae_mse.yaml

In [ ]:
!python -m callaudio.train --config configs/vae/vae_l1.yaml

In [ ]:
!python -m callaudio.train --config configs/vae/vae_beta4.yaml

In [ ]:
!python -m callaudio.train --config configs/vae/cvae.yaml

In [ ]:
!python -m callaudio.train --config configs/vae/vae_dusha.yaml

## Оценка: реконструкция, линейный зонд, судья, интерполяция

In [ ]:
import os
for run in ['vae_mse', 'vae_l1', 'vae_beta4', 'cvae_mse']:
    if os.path.exists(f'{RUNS}/{run}/best.pt'):
        !python -m callaudio.evaluate vae --ckpt {RUNS}/{run}/best.pt --pair yes,no --vocoder hifigan
!python -m callaudio.report --runs $RUNS --out $REPORTS
from IPython.display import Markdown, Image, Audio, display
display(Markdown(open(f'{REPORTS}/vae.md', encoding='utf-8').read()))
display(Image(f'{RUNS}/vae_mse/vae_examples/interpolation_yes_no.png'))
display(Audio(f'{RUNS}/vae_mse/vae_examples/interpolation_yes_no_hifigan.wav'))

## Griffin-Lim и HiFi-GAN

In [ ]:
!python -m callaudio.evaluate vae --ckpt {RUNS}/vae_mse/best.pt --pair yes,no --vocoder griffin_lim
display(Audio(f'{RUNS}/vae_mse/vae_examples/interpolation_yes_no_griffin_lim.wav'))

## Латентное пространство

In [ ]:
!python -m callaudio.viz --run {RUNS}/vae_mse
display(Image(f'{RUNS}/vae_mse/viz/umap_evolution.gif'))